# Workspace for Fourier Tasks

- By: (redacted), 2025
- Corresponding paper (redacted), 2026
- Licence: MIT
- [Click here for the Fourier source repository (redacted)](https://fake.url)

---

## Background

The corpus file consists of eight gong cycle melodies, called 'neliti'
from the traditional repertoire of Gamelan Gong Kebyar,
selected by Tenzer (2000) as representative.

The melodies are:
- Each of length 16.
- Notated by an abbreviated Balinese solfége for the five-note scale,
  - In this abbreviation, five vowels (aeiou) stand in for the note names 'dong', 'deng', 'dung', 'dang', and 'ding'.
  - The corpus uses the minimum distinct description, so aeiou only, omitting the d, n and g.

## Explore

Use the cells below to explore this data and the fact stated above before we begin the tasks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Chose a path (local or remote) and import the data:

In [ ]:
# If you have the whole repo stored locally then use this (and comment out the alternative below)
path_to_data = "./data/"

# If not, use the URL:
path_to_data = "https://raw.githubusercontent.com/music-computing/fourier/refs/heads/main/data/"

# Either way, the import is the same:
df_raw = pd.read_csv(path_to_data + "neliti_length16.csv", header=0)

In [ ]:
# Each row: melody name followed by 16 note tokens (one of a e i o u)
df_raw.head()

---

## Task

Tasks:
1. Convert each melody into indicator vectors for five rhythms, one for each note of the scale (represented by aeiou).
2. Take the DFT of each rhythm and convert to polar coordinates.
3. For each melody, sum the magnitudes across the five rhythms.
4. Plot the summed spectrum for each melody and find the peak coefficient.

Questions:
- Do the spectra peak at some coefficients more often than others?
- Are there any coefficients consistently larger than others across the whole corpus?
- There is one coefficient, $\hat{x}_1$, that is consistently small across the corpus. Why is that the case?
- You converted to polar before summing magnitudes across the five rhythms for each melody. What would happen if you summed the complex coefficients before converting to polar?

---

## Workspace

In [ ]:
# Task 1: convert to per-vowel rhythms


In [ ]:
# Task 2: DFT of each rhythm to polar coordinates


In [ ]:
# Task 3: for each melody, sum the magnitudes across all five rhythms


In [ ]:
# Task 4: plot the summed spectrum for each melody; mark the peak coefficient


---

## Solutions

### Task 1 — Indicator vectors

The `df_raw` imported above contains one row per melody.

Below, we take each vowel `v` to a corresponding `v_vector`,
storing a binary tuple (indicator vector) of length 16
where each position is 1 if the relevant note is present and 0 otherwise.

As each position in the source has exactly one vowel,
the resulting vowel vectors are mutually exclusive and collectively exhaustive:
at every position exactly one vector is 1 and the rest are 0, so their element-wise sum is the all-ones vector.

In [ ]:
VOWELS = ["a", "e", "i", "o", "u"]
NOTE_NAMES = {"a": "dong", "e": "deng", "i": "dung", "o": "dang", "u": "ding"}

# Build indicator-vector representation
rows = []
for _, row in df_raw.iterrows():
    name = row.iloc[0]
    series = row.iloc[1:].dropna().tolist()
    record = {"name": name}
    for v in VOWELS:
        record[f"{v}_vector"] = tuple(1 if x == v else 0 for x in series)
    rows.append(record)

df = pd.DataFrame(rows)
# Show name and "a_vector", for example:
df[["name", "a_vector"]]

### Task 2 — DFT and polar form

As discussed earlier in the paper and in the corresponding notebooks:

1.

For a length-$N$ sequence $x$, the DFT is
$$\hat{x}_k = \sum_{t=0}^{N-1} x_t \, e^{-2\pi i k t / N}, \quad k = 0, 1, \ldots, N-1.$$

2.

Each complex coefficient $\hat{x}_k$ can be written in polar form as
$$\hat{x}_k = |\hat{x}_k|\, e^{i\,\phi_k},$$
where the **magnitude** $|\hat{x}_k|$ captures how much of frequency $k$ is present,
and the **phase** $\phi_k$ captures where in the cycle that frequency is anchored.

In [ ]:
# We make a basic function here to support some reuse later.

def dft_polar(vec):
    """Return (magnitudes, phases) for the DFT of a binary sequence."""
    X = np.fft.fft(vec)
    return np.abs(X), np.angle(X)

# Demonstration on the first melody's 'a' rhythm
demo_name = df.iloc[0]["name"]
demo_vec  = df.iloc[0]["a_vector"]
mags, phases = dft_polar(demo_vec)

In [ ]:
print(f"Melody : {demo_name}")
print(f"a_vector : {demo_vec}")
print(f"Magnitudes (k=0..7): {np.round(mags[:8], 3)}")

### Task 3 — Summed magnitude spectrum

For each melody, sum the DFT magnitudes across the five note-rhythms:
$$M_k = \sum_{v \in \{a,e,i,o,u\}} |\hat{x}^{(v)}_k|.$$

$k=0$ gives $M_0 = N = 16$ for every melody (DC values equal note counts, which sum to $N$). The DFT of a real sequence is also conjugate-symmetric, so $k=9{-}15$ mirror $k=7{-}1$ and carry no new information. We therefore only need $k=1{-}8$.

In [ ]:
N = 16
melody_spectra = {}   # name to length-N (here N=16) array of summed magnitudes

for _, row in df.iterrows():
    summed = np.zeros(N)
    for v in VOWELS:
        mag, _ = dft_polar(row[f"{v}_vector"])
        summed += mag
    melody_spectra[row["name"]] = summed

### Task 4 — Plot and peak coefficients

In [ ]:
ks = np.arange(1, 9)   # k from 1 to 8, exclude DC (k=0) and mirror (k from 9 to 15)

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=True)
axes = axes.flatten()
peak_coefs = {}

for ax, (name, mags) in zip(axes, melody_spectra.items()):
    vals = mags[1:9]
    peak = int(np.argmax(vals)) + 1
    peak_coefs[name] = peak
    colors = ["crimson" if k == peak else "steelblue" for k in ks]
    ax.bar(ks, vals, color=colors, edgecolor="white", linewidth=0.4)
    ax.set_title(name, fontsize=7.5)
    ax.set_xlabel("k", fontsize=7)
    ax.set_xticks(ks)

axes[0].set_ylabel("Summed magnitude", fontsize=8)
fig.suptitle("DFT Magnitude Spectra — Gamelan Neliti Corpus (k=1–8)", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print("Peak coefficients:")
for name, k in peak_coefs.items():
    print(f"  k={k}  {name}")

---

### Question answers

1. Do the spectra peak at some coefficients more often than others?
- Yes. Half (4) of the eight melodies peak at $k=8$, with the remaining three peaking at 6 (twice) and 5 and 7 (one each).
- The corpus leans towards the upper end of the non-redundant range (up to 8).
2. Are there any coefficients consistently larger than others?
- $k=8$ is the most consistently prominent.
- This corresponds to a period of 2 units: the 16-element cycle dividing into two equal halves.
- That is, melodies alternating notes (on, off, on, off, ... ) on the grid.
3. Why is $\hat{x}_1$ consistently small?
- A large $|\hat{x}_1|$ requires a note to be concentrated at one end of the 16-beat cycle and absent at the other.
- That would mean the absence of internal repetition.
- Instead, the corpus melodies have internal phrase structure (see Q2) and more balanced note distributions across the cycle.
- $k=1$ is never the largest, sometimes (2x) the smallest, and overall tends toward the lower half of the spectrum
- Here's a double demonstration of the rank: first _which is lowest_, and second _where $k=1$ falls_:

In [ ]:
for mags in melody_spectra.values():
    # assert
    print(np.argmin(mags[1:9]))

In [ ]:
{name: sorted(mags[1:9]).index(mags[1]) + 1 for name, mags in melody_spectra.items()}

4. What happens if you sum the complex coefficients before converting to polar?
- The result is exactly 0 at every $k \neq 0$, and for every melody.
- The five indicator vectors sum to the all-ones vector, whose DFT is 0 at all non-0 frequencies.
- Summing the complex coefficients is equivalent to taking the DFT of that sum:
$$\sum_v \hat{x}^{(v)}_k = \widehat{\mathbf{1}}_k = 0 \quad \forall\, k \neq 0.$$
- Summing magnitudes first avoids this collapse because $|\hat{x}^{(v)}_k|$ discards phase before aggregating.

In [ ]:
# Demonstration: complex-first sum is identically zero at k != 0
demo_row = df.iloc[0]
complex_sum = sum(np.fft.fft(demo_row[f"{v}_vector"]) for v in VOWELS)

print(f"Melody: {demo_row['name']}")
print(f"{'k':>3}  {'|complex sum|':>15}  {'summed magnitudes':>18}")
for k in range(1, 9):
    print(f"{k:>3}  {abs(complex_sum[k]):>15.3f}  {melody_spectra[demo_row['name']][k]:>18.3f}")

End

---